# MOVIMIENTO DE UNA CAJA

In [1]:
import pybullet as p
import pybullet_data
import numpy as np
import time
import matplotlib.pyplot as plt
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import train_test_split

In [2]:
#para los botones
import ipywidgets as widgets
from IPython.display import display

In [3]:
p.connect(p.GUI)
p.setAdditionalSearchPath(pybullet_data.getDataPath())

#Gravedad
p.setGravity(0, 0, -9.8)

plane_id = p.loadURDF("plane.urdf")
robot_id = p.loadURDF(
    "kuka_iiwa/model.urdf",
    useFixedBase = True
)

num_joints = p.getNumJoints(robot_id)
print("Número de articulaciiones: ", num_joints)

for i in range(num_joints):
    info = p.getJointInfo( robot_id, i)
    print(
        i,
        "->",
        info[1].decode("utf-8"),
        "| tipo:",
        info[2]
    )

Número de articulaciiones:  7
0 -> lbr_iiwa_joint_1 | tipo: 0
1 -> lbr_iiwa_joint_2 | tipo: 0
2 -> lbr_iiwa_joint_3 | tipo: 0
3 -> lbr_iiwa_joint_4 | tipo: 0
4 -> lbr_iiwa_joint_5 | tipo: 0
5 -> lbr_iiwa_joint_6 | tipo: 0
6 -> lbr_iiwa_joint_7 | tipo: 0


In [4]:
NUM_JOINTS_IA = 6
end_effector = 6


# CREAR CAJA

In [5]:
# ==========================================
# CREAR CAJA
# ==========================================

np.random.seed(42)

box_start = [
    np.random.uniform(0.25, 0.55),    # X
    np.random.uniform(-0.30, 0.30),   # Y
    0.025                             # Z
]

box_id = p.loadURDF(
    "cube_small.urdf",
    basePosition=box_start,
    useFixedBase=False
)

print("Caja creada en:")
print(
    f"X = {box_start[0]:.3f}, "
    f"Y = {box_start[1]:.3f}, "
    f"Z = {box_start[2]:.3f}"
)

Caja creada en:
X = 0.362, Y = 0.270, Z = 0.025


In [6]:
# ==========================================
# CREAR POSICIÓN DE DESTINO
# ==========================================

target_destination = [
    np.random.uniform(0.25, 0.55),    # X
    np.random.uniform(-0.30, 0.30),   # Y
    0.025                             # Z
]

print("\nDestino de la caja:")
print(
    f"X = {target_destination[0]:.3f}, "
    f"Y = {target_destination[1]:.3f}, "
    f"Z = {target_destination[2]:.3f}"
)


Destino de la caja:
X = 0.470, Y = 0.059, Z = 0.025


In [7]:
# Crear un marcador visual para el destino
target_marker = p.loadURDF(
    "cube_small.urdf",
    basePosition=target_destination,
    useFixedBase=True
)

print("Marcador de destino creado.")

Marcador de destino creado.


## IA

In [8]:
datos_x = []
datos_y = []
np.random.seed(42)
num_samples = 2000

for _ in range(num_samples):
    #posición aleatoria dentro del espacio de trabajo
    target_pos = [
        np.random.uniform(0.1, 0.7),
        np.random.uniform(-0.5, 0.5),
        np.random.uniform(0.0, 0.8)
    ]
    #Cinemática inversa
    joint_angles = p.calculateInverseKinematics(
        robot_id,
        endEffectorLinkIndex = end_effector,
        targetPosition = target_pos
    )
    #Guardar posición
    datos_x.append(target_pos)
    datos_y.append(joint_angles[:6])

data_x = np.array(datos_x)
data_y = np.array(datos_y)

print(f'Datos generados: {len(data_x)} conjuntos de entrenamiento')
print(f' Entrada: {data_x.shape} (x, y, z)')
print(f' Salida: {data_y.shape} 6 grados de libertad')

Datos generados: 2000 conjuntos de entrenamiento
 Entrada: (2000, 3) (x, y, z)
 Salida: (2000, 6) 6 grados de libertad


In [9]:
x_train, x_test, y_train, y_teset = train_test_split(
    data_x,
    data_y,
    test_size= 0.2,
    random_state= 42
)
print("Datos para entrenamiento: ", x_train.shape)
print("Datos para prueba: ", x_test.shape)

Datos para entrenamiento:  (1600, 3)
Datos para prueba:  (400, 3)


In [10]:
nn = MLPRegressor(
    hidden_layer_sizes=(100, 80, 60),
    activation="relu",
    solver = "adam",
    max_iter= 500,
    random_state= 42,
    verbose= False
)
nn.fit(x_train, y_train)

trains_score = nn.score(x_train, y_train)
test_score = nn.score(x_test, y_teset)

print(f"R² entrenamiento: {trains_score:.4f}")
print(f"R² prueba:        {test_score:.4f}")


R² entrenamiento: 0.4742
R² prueba:        0.6133


### MOVER EL ROBOT

In [11]:
emergency_stop = False
robot_running = False

In [12]:
def mover_robot(angulos, pasos = 300):
    fps = 60
    dt = 1 / fps
    global emergency_stop
    
    for _ in range(pasos):
        if emergency_stop:
            #Un paro de  emergencia hace que el robot pare en ese momento, parando todos los motores
            detener_robot()
            print("PARO DE EMERGENCIA ACTIVADO")
            return False
        
        p.setJointMotorControlArray(
            robot_id,
            range(6),
            p.POSITION_CONTROL,
            targetPositions = angulos
        )
        #Mantener la garra en una posicion fija
        p.setJointMotorControl2(
            robot_id,
            6,
            p.POSITION_CONTROL,
            targetPosition = 0
        )
        p.stepSimulation()
        refrescar_ventana()
        time.sleep(dt)
    
    #Si llegó hasta aquí, el movimiento terminó bien
    return True

In [13]:
def detener_robot():
    posiciones_actuales = []
    
    for i in range(6):
        estado = p.getJointState(robot_id, i)
        posiciones_actuales.append(estado[0])
    p.setJointMotorControlArray(
        robot_id,
        range(6),
        p.POSITION_CONTROL,
        targetPositions = posiciones_actuales
    )
    p.stepSimulation()

In [14]:
def crear_nueva_caja():
    global box_id
    global box_start
    global target_destination
    global target_marker
    
    try:
        p.removeBody(box_id)
    except:
        pass
    try:
        p.removeBody(target_marker)
    except:
        pass
    #Nueva caja
    box_start = [
        np.random.uniform(0.25, 0.55),
        np.random.uniform(-0.30, 0.30),
        0.025
    ]
    
    box_id = p.loadURDF(
        "cube_small.urdf",
        basePosition = box_start,
        useFixedBase = False
    )
    #Nuevo Destino
    target_destination = [
        np.random.uniform(0.25, 0.55),
        np.random.uniform(-0.30, 0.30),
        0.025
    ]
    target_marker = p.loadURDF(
        "cube_small.urdf",
        basePosition = target_destination,
        useFixedBase = True
    )
    print("\nNUEVO DESTINO")
    print(
        f"Caja: x= {box_start[0]:.3f}, "
        f"y = {box_start[1]:.3f}"
    )

In [15]:
import subprocess
import sys
import json

proceso_ventana = None
ultima_posicion = [0, 0, 0]
ultimos_angulos = np.zeros(6)

#Código de la ventana, se ejecuta en un proceso aparte
CODIGO_VENTANA = """
import sys, json
import tkinter as tk

d = json.loads(sys.argv[1])
w = tk.Tk()
w.title('Estado del robot')
w.configure(bg='#1e1e1e', padx=20, pady=15)
w.geometry('+60+60')
w.resizable(False, False)
w.attributes('-topmost', True)

def etiqueta(texto, fuente, color, **kw):
    tk.Label(w, text=texto, font=fuente, fg=color, bg='#1e1e1e').pack(**kw)

etiqueta(d['estado'], ('Arial', 20, 'bold'), d['color'], pady=(0, 10))
p = d['posicion']
etiqueta(
    f'OBJETIVO: x = {p[0]:.3f}   y = {p[1]:.3f}   z = {p[2]:.3f}',
    ('Consolas', 12), '#ffd84d', pady=(0, 8)
)
etiqueta('ANGULOS PREDICHOS POR LA RED', ('Arial', 11, 'bold'), 'white')
for i, rad in enumerate(d['angulos']):
    grados = rad * 180 / 3.141592653589793
    etiqueta(
        f'J{i+1} = {grados:8.2f} deg   ({rad:6.3f} rad)',
        ('Consolas', 12), 'white', anchor='w'
    )

def al_frente():
    w.lift()
    w.attributes('-topmost', True)
    w.after(300, al_frente)

al_frente()
w.mainloop()
"""

def rgb_a_hex(rgb):
    return "#%02x%02x%02x" % tuple(int(c * 255) for c in rgb)

def refrescar_ventana():
    #Ya no se necesita, se deja para que mover_robot no cambie
    pass

def cerrar_ventana():
    global proceso_ventana
    if proceso_ventana is not None:
        proceso_ventana.terminate()
        proceso_ventana = None

def mostrar_prediccion(posicion, angulos):
    #Solo guarda los valores, se dibujan cuando cambia el estado
    global ultima_posicion
    global ultimos_angulos
    ultima_posicion = [float(v) for v in posicion]
    ultimos_angulos = np.array(angulos)

def mostrar_estado(texto, color = [0, 1, 0]):
    global proceso_ventana
    
    #Cerrar la ventana anterior
    cerrar_ventana()
    
    datos = {
        "estado": texto,
        "color": rgb_a_hex(color),
        "posicion": ultima_posicion,
        "angulos": [float(a) for a in ultimos_angulos]
    }
    
    #Abrir la nueva
    proceso_ventana = subprocess.Popen(
        [sys.executable, "-c", CODIGO_VENTANA, json.dumps(datos)],
        stderr = subprocess.PIPE,
        text = True
    )
    
    #Si la ventana no pudo abrir, muestra el motivo
    time.sleep(0.3)
    if proceso_ventana.poll() is not None:
        print("NO SE PUDO ABRIR LA VENTANA:")
        print(proceso_ventana.stderr.read())
        proceso_ventana = None

In [16]:
def predecir_angulos(posicion):
    angulos = nn.predict([posicion])[0]
    return angulos

In [17]:
def ejecución():
    global emergency_stop
    global robot_running
    global constrain_id
    
    if robot_running:
        print("EL PROCESO SE ESTA EJECUTANDO")
        return
    emergency_stop = False
    robot_running = True
    
    print("=" * 60)
    print("PROCESO INICIADO")
    print("=" * 60)
    
    #IR POR LA CAJA
    altura_segura = 0.20
    posicion_sobre_caja = [
        box_start[0],
        box_start[1],
        box_start[2] + altura_segura
    ]
    print("POSICION SOBRE LA CAJA: ")
    print(posicion_sobre_caja)
    angulos = predecir_angulos(posicion_sobre_caja)
    print("\nIR POR LA CAJA")
    if not mover_robot(angulos, 300):
        robot_running = False
        return
    
    #Bajar
    posicion_agarre = [
    box_start[0],
    box_start[1],
    box_start[2] + 0.08
    ]
    angulos= predecir_angulos(
        posicion_agarre
    )
    print("Bajando a la caja")
    if not mover_robot(angulos, 250):
        robot_running = False
        return
    
    #Agarre
    print("AGARRANDO CAJA")
    
    constrain_id = p.createConstraint(
        parentBodyUniqueId = robot_id,
        parentLinkIndex = end_effector,
        childBodyUniqueId = box_id,
        childLinkIndex = -1,
        jointType = p.JOINT_FIXED,
        jointAxis = [0, 0, 0],
        parentFramePosition = [0, 0, 0],
        childFramePosition = [0, 0, -0.02]
    )
    time.sleep(1/60)
    
    #Levantar
    levantar = [
    box_start[0],
    box_start[1],
    0.45
    ]
    angulos= predecir_angulos(
        levantar
    )
    print("LEVANTANDO CAJA")
    if not mover_robot(angulos, 300):
        robot_running = False
        return
    
    #Transportar
    posicion_destino = [
    target_destination[0],
    target_destination[1],
    0.45
    ]
    angulos = predecir_angulos(
        posicion_destino
    )
    print("MOVIENDO LA CAJA")
    if not mover_robot(angulos, 400):
        robot_running = False
        return
    
    #Bajar
    bajar_caja = [
    target_destination[0],
    target_destination[1],
    0.12
    ]
    angulos = predecir_angulos(
        bajar_caja
    )
    print("BAJANDO CAJA")
    if not mover_robot(angulos, 300):
        robot_running = False
        return
    
    #Soltar
    print("SOLTANDO CAJA")
    try:
        p.removeConstraint(constrain_id)
    except:
        pass
    time.sleep(1/60)
    
    robot_running = False
    
    print("=" * 60)
    print("PROCESO TERMINADO")
    print("=" * 60)

In [18]:
def paro_emergencia(b):
    global emergency_stop
    emergency_stop = True
    detener_robot()
    print("\nPARO DE EMERGENCIA")

In [19]:
def reiniciar(b):
    global emergency_stop
    global robot_running
    print("\nREINICIANDO")
    emergency_stop = True
    detener_robot()
    robot_running = False
    
    time.sleep(0.5)
    crear_nueva_caja()
    emergency_stop = False
    
    print("SISTEMA REINICIADO")

# BOTONES

In [20]:
botoninicio = widgets.Button(
    description="START",
    button_style = "success",
    layout = widgets.Layout(
        width = "180px",
        height = "50px"
    )
)

boton_emergencia = widgets.Button(
    description = "PARO DE EMERGENCIA",
    button_style = "danger",
    layout = widgets.Layout(
        width = '220px',
        height = "50 px"
    )
)

restart = widgets.Button(
    description = "RESTART",
    button_style = "info",
    layout = widgets.Layout(
        width = "180px",
        height = "50px"
    )
)

botoninicio.on_click(
    lambda b: ejecución()
)
boton_emergencia.on_click(
    paro_emergencia
)
restart.on_click(
    reiniciar
)

display(
    widgets.HBox([
        botoninicio,
        boton_emergencia,
        restart
    ])
)

In [21]:
test_position = [0.35, 0.0, 0.5]
predicted_angles = predecir_angulos(test_position)
print("Posición objetivo: ")
print(test_position)
print("\nANGULOS PREDICHOS: ")
print(predicted_angles)
print("\nANGULOS EN GRADOS: ")
print(np.degrees(predicted_angles))

Posición objetivo: 
[0.35, 0.0, 0.5]

ANGULOS PREDICHOS: 
[ 7.39609637e-03  6.18356292e-02 -1.71921556e-03 -2.11048507e+00
  4.52852800e-03  4.70539614e-01]

ANGULOS EN GRADOS: 
[ 4.23765107e-01  3.54292058e+00 -9.85037957e-02 -1.20921887e+02
  2.59465542e-01  2.69599340e+01]
